In [1]:
!pip install torch
!pip install --verbose --no-cache-dir torch-scatter
!pip install --verbose --no-cache-dir torch-sparse
!pip install --verbose --no-cache-dir torch-cluster
!pip install torch-geometric
!pip install tensorboardX
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!upzip ngrok-stable-linux-amd64.zip

In [2]:
# Created following to study materials and lessons of Lindsey AI (YouTube Channel Link/Source: https://www.youtube.com/@lindseyai4843)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import sklearn.metrics as metrics

In [5]:
BATCH_SIZE = 32

In [3]:
transform = transforms.Compose([transforms.ToTensor()])

In [4]:
trainset = torchvision.datasets.MNIST(root = './data', train = True, download = True, transform = transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 13.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 339kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.19MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.50MB/s]


In [9]:
trainloader = torch.utils.data.DataLoader(trainset, batch_size = BATCH_SIZE, shuffle = True, num_workers = 2)

In [10]:
testset = torchvision.datasets.MNIST(root = './data', train = False, download = True, transform = transform)

In [12]:
testloader = torch.utils.data.DataLoader(testset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2)

In [14]:
print(len(trainset))

60000


In [16]:
class MyModel(nn.Module):
  def __init__(self):
    super(MyModel, self).__init__()

    self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 32, kernel_size = 3)
    self.d1 = nn.Linear(26 * 26 * 32, 128)
    self.d2 = nn.Linear(128, 10)

  def forward(self, x):
    x = self.conv1(x)
    x = F.relu(x)
    x = x.flatten(start_dim = 1)
    x = self.d1(x)
    x = F.relu(x)
    logits = self.d2(x)
    out = F.softmax(logits, dim = 1)
    return out

In [17]:
learning_rate = 0.001
num_epochs = 5

In [23]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = MyModel()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [34]:
# for epoch in range(num_epochs):
#     train_running_loss = 0.0
#     train_accuracy_score = 0.0
#     total_batches = len(trainloader)

#     print(f'Початок епохи {epoch}...')

#     for i, (images, labels) in enumerate(trainloader):
#         images = images.to(device)
#         labels = labels.to(device)
#         logits = model(images)
#         loss = criterion(logits, labels)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         train_running_loss += loss.detach().item()
#         train_accuracy_score += (torch.argmax(logits, 1).flatten() == labels).type(torch.float).mean().item()

#         if (i + 1) % 100 == 0:
#             print(f'[Прогрес] Оброблено батчів: {i + 1} з {total_batches}')

#     print(f'Кінець епохи {epoch} | Loss: {train_running_loss / total_batches:.4f} | Train Accuracy: {train_accuracy_score / total_batches:.2f} ===\n')

In [28]:
test_accuracy_score = 0.0

for i, (images, labels) in enumerate(testloader, 0):
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    test_accuracy_score += (torch.argmax(outputs, 1).flatten() == labels).type(torch.float).mean().item()

print(f'Test Accuracy: %.2f' % (test_accuracy_score / len(testloader)))

Test Accuracy: 0.81
